<span style = "font-family: Verdana; font-size: 20px">

#### **Data Exploration and Cleaning**
</span>

In [80]:
import pandas as pd
import numpy as np

In [81]:
data = pd.read_csv("data/modified/nonempty_pokemon.csv", keep_default_na=False, dtype = {'ID': str})

In [82]:
combats = pd.read_csv("data/raw_data/combats.csv" #, dtype = {'First_pokemon': str, 'Second_pokemon': str, 'Winner': str}
                      )

<span style = "font-family: Verdana; font-size: 20px">

#### **I. Pokémon dataset**
</span>

In [83]:
def describe_feature(data, feat):
    print(f"- Type: {data.loc[:, feat].dtype}")
    print(f"- First rows:\n{data.loc[:, feat].head(5)}")
    print(f"- Last rows:\n{data.loc[:, feat].tail(5)}")
    print(f"- Number of missing values: {data.loc[:, feat].isna().sum()}")
    print(data.loc[:, feat].dtype)
    print(f"- Number of distinct values: {data.loc[:, feat].nunique()}")
    with pd.option_context('display.max_rows', None):
        print(f"- Unique value counts:\n{data.loc[:, feat].value_counts()}")
    if data.loc[:, feat].dtype in ['int64', 'float64']:
        print(data.describe()[feat])
    else:
        print(f"- Unique values: {data.loc[:, feat].unique()}")


def remove_first_last_letter(series, char):
    series = pd.Series([sublist[1:-1] if len(sublist) >= 3 else sublist 
                      for sublist in series])
    return series.str[1:-1].str.split(f"\'{char} \'")

def get_first_element(series):
    def process_list_first(lst):
        return lst[0]
    
    return series.apply(process_list_first)

def get_second_element(series):
    def process_list_second(lst):
        if len(lst) >= 2:
            return lst[1]
        elif len(lst) == 1:
            return 'None'
    
    return series.apply(process_list_second)


In [84]:
print("Pokémon dataset")
print(f"Number of rows: {data.shape[0]}, number of columns: {data.shape[1]}")
print(f"Column names: {data.columns}")
print(f"Number of missing values: {data.isna().sum().sum()}")

Pokémon dataset
Number of rows: 800, number of columns: 48
Column names: Index(['ID', 'Name', 'Type 1', 'Type 2', 'Abilities', 'HiddenAbility',
       'Generation', 'Hp', 'Attack', 'Defense', 'SpecialAttack',
       'SpecialDefense', 'Speed', 'TotalStats', 'Weight', 'Height',
       'GenderProbM', 'Category', 'CatchRate', 'EggCycles', 'EggGroup',
       'LevelingRate', 'BaseFriendship', 'IsLegendary', 'IsMythical',
       'IsUltraBeast', 'HasMega', 'EvoStage', 'TotalEvoStages', 'PreevoName',
       'DamageFromNormal', 'DamageFromFighting', 'DamageFromFlying',
       'DamageFromPoison', 'DamageFromGround', 'DamageFromRock',
       'DamageFromBug', 'DamageFromGhost', 'DamageFromSteel', 'DamageFromFire',
       'DamageFromWater', 'DamageFromGrass', 'DamageFromElectric',
       'DamageFromPsychic', 'DamageFromIce', 'DamageFromDragon',
       'DamageFromDark', 'DamageFromFairy'],
      dtype='object')
Number of missing values: 0


In [85]:
pd.options.display.max_columns = None
data.head(5)

,ID,Name,Type 1,Type 2,Abilities,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,EggGroup,LevelingRate,BaseFriendship,IsLegendary,IsMythical,IsUltraBeast,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy
0,1,Bulbasaur,Grass,Poison,['Overgrow'],['Chlorophyll'],I,45.0,49.0,49.0,65.0,65.0,45.0,318.0,6.9,0.7,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,0,1,3,No Preevolution,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
1,2,Ivysaur,Grass,Poison,['Overgrow'],['Chlorophyll'],I,60.0,62.0,63.0,80.0,80.0,60.0,405.0,13.0,1.0,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,0,2,3,Bulbasaur,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
2,3,Venusaur,Grass,Poison,['Overgrow'],['Chlorophyll'],I,80.0,82.0,83.0,100.0,100.0,80.0,625.0,100.0,2.0,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,1,3,3,Ivysaur,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5
3,4,Mega Venusaur,Grass,Poison,['Thick Fat'],[],I,80.0,100.0,123.0,122.0,120.0,80.0,625.0,155.5,2.4,0.875,Seed Pokémon,45,20,['Monster' 'Grass'],Medium Slow,70,0,0,0,1,3,3,Venusaur,1.0,0.5,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.25,0.5,2.0,1.0,1.0,1.0,0.5
4,5,Charmander,Fire,None,['Blaze'],['Solar Power'],I,39.0,52.0,43.0,60.0,50.0,65.0,309.0,8.5,0.6,0.875,Lizard Pokémon,45,20,['Monster' 'Dragon'],Medium Slow,70,0,0,0,0,1,3,No Preevolution,1.0,1.0,1.0,1.0,2.0,2.0,0.5,1.0,0.5,0.5,2.0,0.50,1.0,1.0,0.5,1.0,1.0,0.5


<span style = "font-family: Verdana; font-size: 20px">

##### **ID**
</span>

In [86]:
describe_feature(data, 'ID')

- Type: object
- First rows:
0    1
1    2
2    3
3    4
4    5
Name: ID, dtype: object
- Last rows:
795    796
796    797
797    798
798    799
799    800
Name: ID, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 800
- Unique value counts:
ID
1      1
538    1
528    1
529    1
530    1
531    1
532    1
533    1
534    1
535    1
536    1
537    1
539    1
2      1
540    1
541    1
542    1
543    1
544    1
545    1
546    1
547    1
548    1
549    1
527    1
526    1
525    1
524    1
503    1
504    1
505    1
506    1
507    1
508    1
509    1
510    1
511    1
512    1
513    1
514    1
515    1
516    1
517    1
518    1
519    1
520    1
521    1
522    1
523    1
550    1
551    1
552    1
577    1
579    1
580    1
581    1
582    1
583    1
584    1
585    1
586    1
587    1
588    1
589    1
590    1
591    1
592    1
593    1
594    1
595    1
596    1
597    1
598    1
599    1
578    1
576    1
553    1
575    1
554    1
555    1
556 

<span style = "font-family: Verdana; font-size: 20px">

Mỗi Pokémon có một số ID riêng biệt duy nhất, các dạng tiến hoá cũng được xem như một Pokémon riêng biệt với số ID riêng.

<span style = "font-family: Verdana; font-size: 20px">

##### **Name**
</span>

In [87]:
describe_feature(data, 'Name')

- Type: object
- First rows:
0        Bulbasaur
1          Ivysaur
2         Venusaur
3    Mega Venusaur
4       Charmander
Name: Name, dtype: object
- Last rows:
795           Diancie
796      Mega Diancie
797    Hoopa Confined
798     Hoopa Unbound
799         Volcanion
Name: Name, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 800
- Unique value counts:
Name
Bulbasaur                    1
Uxie                         1
Mega Gallade                 1
Probopass                    1
Dusknoir                     1
Froslass                     1
Rotom                        1
Heat Rotom                   1
Wash Rotom                   1
Frost Rotom                  1
Fan Rotom                    1
Mow Rotom                    1
Mesprit                      1
Ivysaur                      1
Azelf                        1
Dialga                       1
Palkia                       1
Heatran                      1
Regigigas                    1
Giratina Alter

<span style = "font-family: Verdana; font-size: 20px">

Mỗi Pokémon cũng có tên riêng, một trong hai cột này đều có thể được dùng để làm primary key.

<span style = "font-family: Verdana; font-size: 20px">

##### **Type**
</span>

<span style = "font-family: Verdana; font-size: 20px">

Để thuận tiện cho việc phân tích, nếu Pokémon chỉ có một loại thì ta sẽ điền giá trị của `Type 1` vào `Type 2`.

In [88]:
describe_feature(data, 'Type 1')
describe_feature(data, 'Type 2')
data.loc[data['Type 2'] == "None", 'Type 2'] = data['Type 1']

- Type: object
- First rows:
0    Grass
1    Grass
2    Grass
3    Grass
4     Fire
Name: Type 1, dtype: object
- Last rows:
795       Rock
796       Rock
797    Psychic
798    Psychic
799       Fire
Name: Type 1, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 18
- Unique value counts:
Type 1
Water       112
Normal       98
Grass        70
Bug          69
Psychic      57
Fire         52
Electric     44
Rock         44
Dragon       32
Ground       32
Ghost        32
Dark         31
Poison       28
Steel        27
Fighting     27
Ice          24
Fairy        17
Flying        4
Name: count, dtype: int64
- Unique values: ['Grass' 'Fire' 'Water' 'Bug' 'Normal' 'Poison' 'Electric' 'Ground'
 'Fairy' 'Fighting' 'Psychic' 'Rock' 'Ghost' 'Ice' 'Dragon' 'Dark' 'Steel'
 'Flying']
- Type: object
- First rows:
0    Poison
1    Poison
2    Poison
3    Poison
4      None
Name: Type 2, dtype: object
- Last rows:
795    Fairy
796    Fairy
797    Ghost
798     Dark
799   

<span style = "font-family: Verdana; font-size: 20px">

Thuộc tính của mỗi Pokémon được phân thành hai loại chính và phụ, tuy nhiên thứ tự của hai thuộc tính này không quan trọng.

<span style = "font-family: Verdana; font-size: 20px">

##### **Abilities**
</span>

In [89]:
for col in ['Abilities', 'HiddenAbility']:
    for abi in data[col]:
        print(abi)

['Overgrow']
['Overgrow']
['Overgrow']
['Thick Fat']
['Blaze']
['Blaze']
['Blaze']
['Tough Claws']
['Drought']
['Torrent']
['Torrent']
['Torrent']
['Mega Launcher']
['Shield Dust']
['Shed Skin']
['Compound Eyes']
['Shield Dust']
['Shed Skin']
['Swarm']
['Adaptability']
['Keen Eye', 'Tangled Feet']
['Keen Eye', 'Tangled Feet']
['Keen Eye', 'Tangled Feet']
['No Guard']
['Run Away', 'Guts']
['Run Away', 'Guts']
['Keen Eye']
['Keen Eye']
['Intimidate', 'Shed Skin']
['Intimidate', 'Shed Skin']
['Static']
['Static']
['Sand Veil']
['Sand Veil']
['Poison Point', 'Rivalry']
['Poison Point', 'Rivalry']
['Poison Point', 'Rivalry']
['Poison Point', 'Rivalry']
['Poison Point', 'Rivalry']
['Poison Point', 'Rivalry']
['Cute Charm', 'Magic Guard']
['Cute Charm', 'Magic Guard']
['Flash Fire']
['Flash Fire']
['Cute Charm', 'Competitive']
['Cute Charm', 'Competitive']
['Inner Focus']
['Inner Focus']
['Chlorophyll']
['Chlorophyll']
['Chlorophyll']
['Effect Spore', 'Dry Skin']
['Effect Spore', 'Dry Skin']


<span style = "font-family: Verdana; font-size: 20px">

Thuộc tính này có chứa một hoặc nhiều thành phần, bao gồm cả các ký tự đặc biệt, cần phải được làm sạch và tách riêng.

In [90]:
data['Abilities'] = remove_first_last_letter(data['Abilities'], ',')
data['HiddenAbility'] = remove_first_last_letter(data['HiddenAbility'], ',')
data['Ability 1'] = get_first_element(data['Abilities'])
data['Ability 2'] = get_second_element(data['Abilities'])
data['HiddenAbility'] = get_first_element(data['HiddenAbility'])
data = data.drop('Abilities', axis=1)

In [91]:
describe_feature(data, 'Ability 1')

- Type: object
- First rows:
0     Overgrow
1     Overgrow
2     Overgrow
3    Thick Fat
4        Blaze
Name: Ability 1, dtype: object
- Last rows:
795      Clear Body
796    Magic Bounce
797        Magician
798        Magician
799    Water Absorb
Name: Ability 1, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 149
- Unique value counts:
Ability 1
Levitate          39
Swift Swim        28
Chlorophyll       25
Pressure          23
Intimidate        22
Keen Eye          20
Sturdy            19
Overgrow          18
Torrent           18
Blaze             18
Swarm             16
Thick Fat         15
Run Away          14
Shed Skin         13
Poison Point      13
Frisk             13
Natural Cure      12
Cute Charm        12
Static            12
Oblivious         12
Guts              12
Inner Focus       11
Synchronize       11
Clear Body        10
Water Absorb      10
Rock Head         10
Hyper Cutter       9
Insomnia           9
Sand Veil          9
Pickup   

In [92]:
describe_feature(data, 'Ability 2')

- Type: object
- First rows:
0    None
1    None
2    None
3    None
4    None
Name: Ability 2, dtype: object
- Last rows:
795    None
796    None
797    None
798    None
799    None
Name: Ability 2, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 112
- Unique value counts:
Ability 2
None                392
Shell Armor          11
Sturdy               11
Inner Focus          10
Sniper                9
Hydration             9
Own Tempo             9
Early Bird            9
Technician            8
Rock Head             8
Infiltrator           7
Keen Eye              7
Flash Fire            7
Ice Body              7
Competitive           7
Magic Guard           7
Super Luck            6
Serene Grace          6
Sticky Hold           6
Rivalry               6
Mold Breaker          6
Sheer Force           6
Klutz                 6
Quick Feet            5
Tangled Feet          5
Moxie                 5
Pickup                5
Frisk                 5
Snow Cloak 

In [93]:
print(data['Ability 1'].unique())
print(data['Ability 2'].unique())
print(data['HiddenAbility'].unique())

['Overgrow' 'Thick Fat' 'Blaze' 'Tough Claws' 'Drought' 'Torrent'
 'Mega Launcher' 'Shield Dust' 'Shed Skin' 'Compound Eyes' 'Swarm'
 'Adaptability' 'Keen Eye' 'No Guard' 'Run Away' 'Intimidate' 'Static'
 'Sand Veil' 'Poison Point' 'Cute Charm' 'Flash Fire' 'Inner Focus'
 'Chlorophyll' 'Effect Spore' 'Pickup' 'Limber' 'Damp' 'Vital Spirit'
 'Water Absorb' 'Synchronize' 'Trace' 'Guts' 'Clear Body' 'Rock Head'
 'Oblivious' 'Shell Armor' 'Magnet Pull' 'Stench' 'Levitate' 'Cursed Body'
 'Shadow Tag' 'Insomnia' 'Hyper Cutter' 'Soundproof' 'Own Tempo'
 'Lightning Rod' 'Natural Cure' 'Early Bird' 'Parental Bond' 'Swift Swim'
 'Illuminate' 'Flame Body' 'Aerilate' 'Mold Breaker' 'Volt Absorb'
 'Immunity' 'Pressure' 'Steadfast' 'Hustle' 'Sturdy' 'Speed Boost'
 'Serene Grace' 'Sand Force' 'Technician' 'Skill Link' 'Magma Armor'
 'Suction Cups' 'Solar Power' 'Sand Stream' 'Pixilate' 'Truant'
 'Wonder Guard' 'Magic Bounce' 'Huge Power' 'Filter' 'Pure Power' 'Plus'
 'Minus' 'Liquid Ooze' 'Rough Skin

<span style = "font-family: Verdana; font-size: 20px">

Thay thế các ô bị trống bằng `None`

In [94]:
data['Ability 1'] = data['Ability 1'].replace(['nan', ''], ['None', 'None'])
data['Ability 2'] = data['Ability 2'].replace(['nan', ''], ['None', 'None'])
data['HiddenAbility'] = data['HiddenAbility'].replace(['nan', ''], ['None', 'None'])
data['Ability 1'] = data['Ability 1'].fillna('None')
data['Ability 2'] = data['Ability 2'].fillna('None')
data['HiddenAbility'] = data['HiddenAbility'].fillna('None')

In [95]:
describe_feature(data, 'HiddenAbility')

- Type: object
- First rows:
0    Chlorophyll
1    Chlorophyll
2    Chlorophyll
3           None
4    Solar Power
Name: HiddenAbility, dtype: object
- Last rows:
795    None
796    None
797    None
798    None
799    None
Name: HiddenAbility, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 135
- Unique value counts:
HiddenAbility
None             155
Sheer Force       17
Unnerve           15
Weak Armor        15
Telepathy         14
Infiltrator       14
Overcoat          14
Regenerator       13
Analytic          12
Rattled           11
Sand Force        11
Swift Swim        10
Damp              10
Hydration          9
Gluttony           9
Inner Focus        9
Insomnia           9
Moxie              8
Sap Sipper         8
Reckless           8
Speed Boost        8
Frisk              8
Rain Dish          8
Friend Guard       8
Run Away           8
Scrappy            8
Hustle             8
Vital Spirit       7
Mold Breaker       7
Shell Armor        7
Pickpo

<span style = "font-family: Verdana; font-size: 20px">

##### **Generation**
</span>

In [96]:
describe_feature(data, 'Generation')

- Type: object
- First rows:
0    I
1    I
2    I
3    I
4    I
Name: Generation, dtype: object
- Last rows:
795    VI
796    VI
797    VI
798    VI
799    VI
Name: Generation, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 6
- Unique value counts:
Generation
I      166
V      165
III    160
IV     121
II     106
VI      82
Name: count, dtype: int64
- Unique values: ['I' 'II' 'III' 'IV' 'V' 'VI']


In [97]:
data['Generation'] = data['Generation'].replace(['I', 'II', 'III', 'IV', 'V', 'VI'], [1, 2, 3, 4, 5, 6]).infer_objects(copy=False)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_14912\1242375916.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Generation'] = data['Generation'].replace(['I', 'II', 'III', 'IV', 'V', 'VI'], [1, 2, 3, 4, 5, 6]).infer_objects(copy=False)


In [98]:
describe_feature(data, 'Generation')

- Type: int64
- First rows:
0    1
1    1
2    1
3    1
4    1
Name: Generation, dtype: int64
- Last rows:
795    6
796    6
797    6
798    6
799    6
Name: Generation, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 6
- Unique value counts:
Generation
1    166
5    165
3    160
4    121
2    106
6     82
Name: count, dtype: int64
count    800.00000
mean       3.32375
std        1.66129
min        1.00000
25%        2.00000
50%        3.00000
75%        5.00000
max        6.00000
Name: Generation, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **HP**
</span>

In [99]:
describe_feature(data, 'Hp')

- Type: float64
- First rows:
0    45.0
1    60.0
2    80.0
3    80.0
4    39.0
Name: Hp, dtype: float64
- Last rows:
795    50.0
796    50.0
797    80.0
798    80.0
799    80.0
Name: Hp, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 94
- Unique value counts:
Hp
60.0     68
50.0     62
70.0     54
65.0     46
80.0     44
75.0     44
40.0     38
45.0     38
55.0     35
100.0    32
90.0     29
95.0     22
85.0     20
35.0     15
30.0     13
105.0    10
78.0      9
110.0     9
79.0      7
68.0      7
91.0      7
62.0      7
44.0      7
20.0      6
59.0      6
38.0      6
58.0      6
74.0      6
64.0      6
76.0      5
72.0      5
67.0      5
106.0     5
86.0      4
125.0     4
71.0      4
108.0     4
150.0     4
73.0      4
61.0      4
41.0      3
83.0      3
66.0      3
43.0      3
115.0     3
49.0      3
103.0     3
63.0      3
120.0     3
54.0      3
89.0      3
130.0     3
48.0      3
77.0      3
69.0      2
53.0      2
57.0      2
39.0      2
46.0 

<span style = "font-family: Verdana; font-size: 20px">

##### **Attack**
</span>

In [100]:
describe_feature(data, 'Attack')

- Type: float64
- First rows:
0     49.0
1     62.0
2     82.0
3    100.0
4     52.0
Name: Attack, dtype: float64
- Last rows:
795    100.0
796    160.0
797    110.0
798    160.0
799    110.0
Name: Attack, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 111
- Unique value counts:
Attack
100.0    41
65.0     39
50.0     37
80.0     36
90.0     32
85.0     32
75.0     32
60.0     32
70.0     31
55.0     30
45.0     26
95.0     26
40.0     21
120.0    21
30.0     20
105.0    18
110.0    17
130.0    15
125.0    14
35.0     13
150.0    10
48.0      9
115.0     9
20.0      8
52.0      7
66.0      7
25.0      7
92.0      7
140.0     7
72.0      7
82.0      6
63.0      6
135.0     6
84.0      5
53.0      5
73.0      5
64.0      5
160.0     5
83.0      4
78.0      4
69.0      4
77.0      4
145.0     4
62.0      4
76.0      4
58.0      4
59.0      3
38.0      3
68.0      3
165.0     3
49.0      3
180.0     3
47.0      3
123.0     3
10.0      3
98.0      3
104.0 

<span style = "font-family: Verdana; font-size: 20px">

##### **Defense**
</span>

In [101]:
describe_feature(data, 'Defense')

- Type: float64
- First rows:
0     49.0
1     63.0
2     83.0
3    123.0
4     43.0
Name: Defense, dtype: float64
- Last rows:
795    150.0
796    110.0
797     60.0
798     60.0
799    120.0
Name: Defense, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 103
- Unique value counts:
Defense
70.0     53
50.0     50
60.0     46
80.0     40
65.0     36
40.0     36
90.0     35
100.0    33
45.0     32
55.0     30
85.0     29
95.0     27
75.0     27
35.0     23
105.0    15
30.0     13
120.0    12
110.0    12
48.0     11
115.0    11
130.0    10
67.0      8
140.0     7
62.0      7
63.0      7
58.0      6
150.0     6
77.0      6
125.0     6
72.0      6
43.0      6
78.0      6
53.0      5
122.0     5
107.0     5
52.0      5
15.0      4
76.0      4
57.0      4
44.0      4
88.0      4
79.0      4
86.0      4
64.0      4
20.0      4
230.0     3
37.0      3
66.0      3
68.0      3
42.0      3
34.0      3
160.0     3
180.0     3
83.0      3
84.0      2
118.0     2
135

<span style = "font-family: Verdana; font-size: 20px">

##### **SpecialAttack**
</span>

In [102]:
describe_feature(data, 'SpecialAttack')

- Type: float64
- First rows:
0     65.0
1     80.0
2    100.0
3    122.0
4     60.0
Name: SpecialAttack, dtype: float64
- Last rows:
795    100.0
796    160.0
797    150.0
798    170.0
799    130.0
Name: SpecialAttack, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 105
- Unique value counts:
SpecialAttack
60.0     51
40.0     49
65.0     44
50.0     38
55.0     35
45.0     33
70.0     30
35.0     29
100.0    28
95.0     28
85.0     26
80.0     25
30.0     24
90.0     22
105.0    20
75.0     18
110.0    16
125.0    14
120.0    14
25.0     11
130.0    11
115.0     9
20.0      8
150.0     8
44.0      8
83.0      8
58.0      7
135.0     7
81.0      6
61.0      6
63.0      5
140.0     5
74.0      5
62.0      5
73.0      4
92.0      4
97.0      4
69.0      4
54.0      4
53.0      4
15.0      4
109.0     4
145.0     4
77.0      3
57.0      3
37.0      3
180.0     3
98.0      3
72.0      3
10.0      3
48.0      3
170.0     3
59.0      3
56.0      3
79.0     

<span style = "font-family: Verdana; font-size: 20px">

##### **SpecialDefense**
</span>

In [103]:
describe_feature(data, 'SpecialDefense')

- Type: float64
- First rows:
0     65.0
1     80.0
2    100.0
3    120.0
4     50.0
Name: SpecialDefense, dtype: float64
- Last rows:
795    150.0
796    110.0
797    130.0
798    130.0
799     90.0
Name: SpecialDefense, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 92
- Unique value counts:
SpecialDefense
80.0     51
50.0     50
55.0     47
65.0     43
60.0     42
70.0     41
75.0     39
90.0     37
45.0     35
85.0     31
40.0     30
95.0     29
100.0    28
30.0     20
105.0    18
35.0     18
110.0    15
120.0    14
25.0     11
63.0     10
115.0    10
48.0      9
130.0     8
56.0      7
20.0      6
150.0     6
107.0     6
42.0      5
86.0      5
135.0     4
71.0      4
82.0      4
41.0      4
52.0      4
79.0      4
61.0      4
87.0      3
77.0      3
43.0      3
69.0      3
44.0      3
72.0      3
81.0      3
154.0     3
140.0     3
96.0      3
83.0      3
66.0      3
62.0      3
54.0      3
37.0      3
67.0      2
128.0     2
39.0      2
57.0   

<span style = "font-family: Verdana; font-size: 20px">

##### **Speed**
</span>

In [104]:
describe_feature(data, 'Speed')

- Type: float64
- First rows:
0    45.0
1    60.0
2    80.0
3    80.0
4    65.0
Name: Speed, dtype: float64
- Last rows:
795     50.0
796    110.0
797     70.0
798     80.0
799     70.0
Name: Speed, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 108
- Unique value counts:
Speed
50.0     46
60.0     43
65.0     36
70.0     36
30.0     35
80.0     34
40.0     32
90.0     32
55.0     30
100.0    30
45.0     29
95.0     27
85.0     27
35.0     22
110.0    16
75.0     16
20.0     15
105.0    12
115.0    11
25.0     10
15.0      9
58.0      8
108.0     7
101.0     7
68.0      6
42.0      6
130.0     6
71.0      6
48.0      6
120.0     5
36.0      5
91.0      5
150.0     5
64.0      5
56.0      5
97.0      5
86.0      5
43.0      5
92.0      4
57.0      4
32.0      4
23.0      4
66.0      4
28.0      4
67.0      4
78.0      4
81.0      4
72.0      4
99.0      4
38.0      3
52.0      3
145.0     3
84.0      3
74.0      3
112.0     3
29.0      3
104.0     3
41

<span style = "font-family: Verdana; font-size: 20px">

##### **TotalStats**
</span>

In [105]:
describe_feature(data, 'TotalStats')

- Type: float64
- First rows:
0    318.0
1    405.0
2    625.0
3    625.0
4    309.0
Name: TotalStats, dtype: float64
- Last rows:
795    700.0
796    700.0
797    680.0
798    680.0
799    600.0
Name: TotalStats, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 196
- Unique value counts:
TotalStats
600.0    34
580.0    26
405.0    25
300.0    19
500.0    19
490.0    17
700.0    17
495.0    16
330.0    15
525.0    14
480.0    13
305.0    13
680.0    12
485.0    12
310.0    10
520.0    10
455.0    10
335.0     9
510.0     9
420.0     9
540.0     9
460.0     9
430.0     8
505.0     8
325.0     8
340.0     8
410.0     8
515.0     8
630.0     8
535.0     8
280.0     7
350.0     7
250.0     7
440.0     7
290.0     7
390.0     7
320.0     7
470.0     7
295.0     6
355.0     6
465.0     6
530.0     6
360.0     6
450.0     6
400.0     5
590.0     5
475.0     5
780.0     5
205.0     5
395.0     5
494.0     5
275.0     5
385.0     4
770.0     4
625.0     4
309.0 

In [106]:
for i in range(data.shape[0]):
    total = data.loc[i, 'Hp'] + data.loc[i, 'Attack'] + data.loc[i, 'Defense'] + data.loc[i, 'SpecialAttack'] + data.loc[i, 'SpecialDefense'] + data.loc[i, 'Speed']
    if not data.at[i, 'TotalStats'] == total:
        print(data.at[i, 'Name'], data.at[i, 'TotalStats'],  total)

Venusaur 625.0 525.0
Charizard 634.0 534.0
Blastoise 630.0 530.0
Beedrill 495.0 395.0
Pidgeot 579.0 479.0
Pikachu 430.0 320.0
Alakazam 600.0 500.0
Gengar 600.0 500.0
Kangaskhan 590.0 490.0
Pinsir 600.0 500.0
Gyarados 640.0 540.0
Eevee 435.0 325.0
Aerodactyl 615.0 515.0
Mewtwo 780.0 680.0
Ampharos 610.0 510.0
Steelix 610.0 510.0
Scizor 600.0 500.0
Heracross 600.0 500.0
Houndoom 600.0 500.0
Tyranitar 700.0 600.0
Sceptile 630.0 530.0
Blaziken 630.0 530.0
Swampert 635.0 535.0
Gardevoir 618.0 518.0
Sableye 480.0 380.0
Mawile 480.0 380.0
Aggron 630.0 530.0
Medicham 510.0 410.0
Manectric 575.0 475.0
Sharpedo 560.0 460.0
Camerupt 560.0 460.0
Altaria 590.0 490.0
Banette 555.0 455.0
Absol 565.0 465.0
Glalie 580.0 480.0
Salamence 700.0 600.0
Metagross 700.0 600.0
Latias 700.0 600.0
Latios 700.0 600.0
Kyogre 770.0 670.0
Groudon 770.0 670.0
Rayquaza 780.0 680.0
Lopunny 580.0 480.0
Garchomp 700.0 600.0
Lucario 625.0 525.0
Abomasnow 594.0 494.0
Gallade 618.0 518.0
Rotom 520.0 440.0
Audino 545.0 445.0

<span style = "font-family: Verdana; font-size: 20px">

Có vẻ như tổng điểm chỉ số của một số các Pokémon có dạng tiến hoá Mega bị lệch so với tổng chỉ số thực tế của chúng, cần phải tính lại cột này.

In [107]:
data['TotalStats'] = data['Hp'] + data['Attack'] + data['Defense'] + data['SpecialAttack'] + data['SpecialDefense'] + data['Speed']

<span style = "font-family: Verdana; font-size: 20px">

##### **Weight**
</span>

In [108]:
describe_feature(data, 'Weight')

- Type: float64
- First rows:
0      6.9
1     13.0
2    100.0
3    155.5
4      8.5
Name: Weight, dtype: float64
- Last rows:
795      8.8
796     27.8
797      9.0
798      9.0
799    195.0
Name: Weight, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 421


- Unique value counts:
Weight
0.3      10
5.0       9
6.5       9
8.5       9
15.0      9
30.0      8
28.0      8
1.0       7
9.0       7
2.0       7
12.0      6
4.0       6
9.5       6
12.5      6
55.0      6
120.0     6
60.0      6
6.0       5
11.5      5
5.5       5
20.0      5
19.5      5
7.5       5
38.0      5
2.1       5
32.0      5
40.5      5
3.5       5
11.0      4
1.2       4
32.5      4
31.5      4
18.0      4
27.0      4
33.0      4
2.5       4
14.0      4
31.0      4
35.0      4
8.0       4
29.0      4
10.5      4
60.8      4
13.0      4
52.0      4
7.0       4
50.5      4
10.0      4
29.5      4
40.0      4
21.0      3
68.0      3
0.1       3
24.0      3
4.5       3
6.6       3
19.0      3
61.0      3
325.0     3
0.6       3
80.0      3
39.5      3
25.0      3
39.0      3
100.0     3
260.0     3
24.5      3
220.0     3
28.5      3
23.5      3
22.0      3
3.0       3
16.0      3
54.0      3
61.5      3
20.5      3
14.5      3
15.5      3
33.5      3
9.9       3
5.8       

<span style = "font-family: Verdana; font-size: 20px">

##### **Height**
</span>

In [109]:
describe_feature(data, 'Height')

- Type: float64
- First rows:
0    0.7
1    1.0
2    2.0
3    2.4
4    0.6
Name: Height, dtype: float64
- Last rows:
795    0.7
796    1.1
797    0.5
798    0.5
799    1.7
Name: Height, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 51
- Unique value counts:
Height
0.6     68
0.5     61
1.0     59
0.4     56
0.3     51
1.2     48
1.5     47
0.8     44
1.1     38
0.7     35
1.4     33
0.9     32
1.3     31
1.7     30
1.6     26
2.0     21
1.8     19
1.9     16
0.2     14
2.1      9
2.2      9
2.5      8
3.0      4
2.3      4
4.5      3
2.7      3
0.1      2
3.2      2
5.0      2
3.5      2
6.5      2
2.4      2
5.2      1
4.2      1
8.8      1
2.9      1
2.8      1
2.6      1
3.3      1
3.7      1
5.4      1
3.8      1
10.8     1
7.0      1
4.0      1
9.8      1
9.2      1
6.2      1
10.5     1
14.5     1
5.8      1
Name: count, dtype: int64
count    800.000000
mean       1.222875
std        1.181041
min        0.100000
25%        0.600000
50%        1

<span style = "font-family: Verdana; font-size: 20px">

##### **GenderProbM**
</span>

In [110]:
describe_feature(data, 'GenderProbM')

- Type: float64
- First rows:
0    0.875
1    0.875
2    0.875
3    0.875
4    0.875
Name: GenderProbM, dtype: float64
- Last rows:
795   -1.0
796   -1.0
797   -1.0
798   -1.0
799   -1.0
Name: GenderProbM, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 8
- Unique value counts:
GenderProbM
 0.500    495
 0.875    110
-1.000     99
 0.000     28
 1.000     24
 0.250     22
 0.750     20
 0.125      2
Name: count, dtype: int64
count    800.000000
mean       0.361875
std        0.546765
min       -1.000000
25%        0.500000
50%        0.500000
75%        0.500000
max        1.000000
Name: GenderProbM, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

GenderProbM cho biết tỷ lệ giống đực của Pokémon, giá trị `-1` biểu thị cho các Pokémon không có giới tính như các Pokémon huyền thoại hay thần thoại. Để khoảng giá trị của cột này liên tục từ 0 đến 1, ta sẽ thay thế các giá trị `-1` này bằng `0.5` và thêm một cột để thể hiện rằng Pokémon đó có giới tính hay không.

In [111]:
data['NoGender'] = 0
data.loc[data['GenderProbM'] == -1, 'NoGender'] = 1
data['GenderProbM'] = data['GenderProbM'].replace([-1], [0.5]).astype('float64')
describe_feature(data, 'GenderProbM')
describe_feature(data, 'NoGender')

- Type: float64
- First rows:
0    0.875
1    0.875
2    0.875
3    0.875
4    0.875
Name: GenderProbM, dtype: float64
- Last rows:
795    0.5
796    0.5
797    0.5
798    0.5
799    0.5
Name: GenderProbM, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 7
- Unique value counts:
GenderProbM
0.500    594
0.875    110
0.000     28
1.000     24
0.250     22
0.750     20
0.125      2
Name: count, dtype: int64
count    800.000000
mean       0.547500
std        0.192377
min        0.000000
25%        0.500000
50%        0.500000
75%        0.500000
max        1.000000
Name: GenderProbM, dtype: float64
- Type: int64
- First rows:
0    0
1    0
2    0
3    0
4    0
Name: NoGender, dtype: int64
- Last rows:
795    1
796    1
797    1
798    1
799    1
Name: NoGender, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 2
- Unique value counts:
NoGender
0    701
1     99
Name: count, dtype: int64
count    800.000000
mean       0.123750
st

<span style = "font-family: Verdana; font-size: 20px">

##### **Category**
</span>

In [112]:
describe_feature(data, 'Category')

- Type: object
- First rows:
0      Seed Pokémon
1      Seed Pokémon
2      Seed Pokémon
3      Seed Pokémon
4    Lizard Pokémon
Name: Category, dtype: object
- Last rows:
795       Jewel Pokémon
796       Jewel Pokémon
797    Mischief Pokémon
798    Mischief Pokémon
799       Steam Pokémon
Name: Category, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 524
- Unique value counts:
Category
Dragon Pokémon           9
Pumpkin Pokémon          8
Flame Pokémon            7
Mushroom Pokémon         6
Bagworm Pokémon          6
Mouse Pokémon            6
Plasma Pokémon           6
Fox Pokémon              5
Balloon Pokémon          5
Seed Pokémon             5
Bat Pokémon              4
Poison Pin Pokémon       4
Mud Fish Pokémon         4
Iron Armor Pokémon       4
Eon Pokémon              4
Fairy Pokémon            4
Psi Pokémon              4
Drill Pokémon            4
Tadpole Pokémon          4
DNA Pokémon              4
Shellfish Pokémon        4
Cocoon Po

<span style = "font-family: Verdana; font-size: 20px">

##### **CatchRate**
</span>

In [113]:
describe_feature(data, 'CatchRate')

- Type: int64
- First rows:
0    45
1    45
2    45
3    45
4    45
Name: CatchRate, dtype: int64
- Last rows:
795    3
796    3
797    3
798    3
799    3
Name: CatchRate, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 33
- Unique value counts:
CatchRate
45     260
3       72
190     70
255     65
75      60
120     54
60      51
90      37
30      20
200     18
225     14
25      11
180     10
50       8
150      8
235      6
100      5
140      4
127      4
65       3
125      3
55       3
170      2
130      2
220      2
70       1
205      1
155      1
145      1
35       1
15       1
160      1
80       1
Name: count, dtype: int64
count    800.000000
mean      94.755000
std       75.716899
min        3.000000
25%       45.000000
50%       60.000000
75%      140.000000
max      255.000000
Name: CatchRate, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **EggCycles**
</span>

In [114]:
describe_feature(data, 'EggCycles')

- Type: int64
- First rows:
0    20
1    20
2    20
3    20
4    20
Name: EggCycles, dtype: int64
- Last rows:
795     25
796     25
797    120
798    120
799    120
Name: EggCycles, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 10
- Unique value counts:
EggCycles
20     449
15     112
25      61
120     53
40      43
30      26
10      21
80      17
35      15
5        3
Name: count, dtype: int64
count    800.000000
mean      28.943750
std       26.414489
min        5.000000
25%       20.000000
50%       20.000000
75%       25.000000
max      120.000000
Name: EggCycles, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **EggGroup**
</span>

In [115]:
describe_feature(data, 'EggGroup')

- Type: object
- First rows:
0     ['Monster' 'Grass']
1     ['Monster' 'Grass']
2     ['Monster' 'Grass']
3     ['Monster' 'Grass']
4    ['Monster' 'Dragon']
Name: EggGroup, dtype: object
- Last rows:
795    []
796    []
797    []
798    []
799    []
Name: EggGroup, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 59
- Unique value counts:
EggGroup
['Field']                     141
[]                             94
['Bug']                        60
['Amorphous']                  50
['Mineral']                    46
['Human-Like']                 39
['Flying']                     36
['Monster']                    21
['Monster' 'Dragon']           20
['Grass']                      19
['Fairy']                      17
['Water 1' 'Field']            17
['Monster' 'Water 1']          16
['Water 3']                    14
['Monster' 'Grass']            14
['Water 1']                    14
['Monster' 'Field']            14
['Field' 'Human-Like']         13
['Wat

<span style = "font-family: Verdana; font-size: 20px">

Cột này sẽ được xử lý giống như với `Type`. Hơn nữa, có một số Pokémon không có nhóm trứng (Pokémon không thể phối giống, hoặc nhóm trứng không được phát hiện), ô trống này sẽ được thay thế bằng giá trị khác.

In [116]:
data['EggGroup'] = data['EggGroup'].str.rstrip(r'\"')
data['EggGroup'] = data['EggGroup'].str.lstrip(r'\"')
data['EggGroup'] = remove_first_last_letter(data['EggGroup'], '')
data['EggGroup1'] = get_first_element(data['EggGroup'])
data['EggGroup2'] = get_second_element(data['EggGroup'])
data.loc[data['EggGroup1'] == '', 'EggGroup1'] = 'No Eggs Discovered'
data.loc[data['EggGroup2'] == 'None', 'EggGroup2'] = 'No Eggs Discovered'

describe_feature(data, 'EggGroup1')
describe_feature(data, 'EggGroup2')
data = data.drop('EggGroup', axis=1)

- Type: object
- First rows:
0    Monster
1    Monster
2    Monster
3    Monster
4    Monster
Name: EggGroup1, dtype: object
- Last rows:
795    No Eggs Discovered
796    No Eggs Discovered
797    No Eggs Discovered
798    No Eggs Discovered
799    No Eggs Discovered
Name: EggGroup1, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 15
- Unique value counts:
EggGroup1
Field                 179
No Eggs Discovered     94
Monster                87
Water 1                74
Bug                    72
Amorphous              50
Mineral                49
Flying                 47
Human-Like             45
Fairy                  32
Grass                  27
Water 2                17
Water 3                14
Dragon                 12
Ditto                   1
Name: count, dtype: int64
- Unique values: ['Monster' 'Bug' 'Flying' 'Field' 'No Eggs Discovered' 'Fairy' 'Grass'
 'Water 1' 'Human-Like' 'Water 3' 'Mineral' 'Amorphous' 'Water 2' 'Ditto'
 'Dragon']
- Type: ob

<span style = "font-family: Verdana; font-size: 20px">

##### **Leveling Rate**
</span>

In [117]:
describe_feature(data, 'LevelingRate')

- Type: object
- First rows:
0    Medium Slow
1    Medium Slow
2    Medium Slow
3    Medium Slow
4    Medium Slow
Name: LevelingRate, dtype: object
- Last rows:
795    Slow
796    Slow
797    Slow
798    Slow
799    Slow
Name: LevelingRate, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 6
- Unique value counts:
LevelingRate
Medium Fast    320
Medium Slow    205
Slow           182
Fast            56
Erratic         23
Fluctuating     14
Name: count, dtype: int64
- Unique values: ['Medium Slow' 'Medium Fast' 'Fast' 'Slow' 'Fluctuating' 'Erratic']


<span style = "font-family: Verdana; font-size: 20px">

##### **Base Frienship**
</span>

In [118]:
describe_feature(data, 'BaseFriendship')

- Type: int64
- First rows:
0    70
1    70
2    70
3    70
4    70
Name: BaseFriendship, dtype: int64
- Last rows:
795     70
796      0
797    100
798    100
799    100
Name: BaseFriendship, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 6
- Unique value counts:
BaseFriendship
70     649
35      80
0       33
100     17
140     11
90      10
Name: count, dtype: int64
count    800.000000
mean      65.462500
std       19.900531
min        0.000000
25%       70.000000
50%       70.000000
75%       70.000000
max      140.000000
Name: BaseFriendship, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **IsLegendary**
</span>

In [119]:
describe_feature(data, 'IsLegendary')

- Type: int64
- First rows:
0    0
1    0
2    0
3    0
4    0
Name: IsLegendary, dtype: int64
- Last rows:
795    0
796    0
797    0
798    0
799    0
Name: IsLegendary, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 2
- Unique value counts:
IsLegendary
0    749
1     51
Name: count, dtype: int64
count    800.00000
mean       0.06375
std        0.24446
min        0.00000
25%        0.00000
50%        0.00000
75%        0.00000
max        1.00000
Name: IsLegendary, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **IsMythical**
</span>

In [120]:
describe_feature(data, 'IsMythical')

- Type: int64
- First rows:
0    0
1    0
2    0
3    0
4    0
Name: IsMythical, dtype: int64
- Last rows:
795    1
796    1
797    1
798    1
799    1
Name: IsMythical, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 2
- Unique value counts:
IsMythical
0    776
1     24
Name: count, dtype: int64
count    800.000000
mean       0.030000
std        0.170694
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: IsMythical, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **IsUltraBeast**
</span>

In [121]:
describe_feature(data, 'IsUltraBeast')

- Type: int64
- First rows:
0    0
1    0
2    0
3    0
4    0
Name: IsUltraBeast, dtype: int64
- Last rows:
795    0
796    0
797    0
798    0
799    0
Name: IsUltraBeast, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 1
- Unique value counts:
IsUltraBeast
0    800
Name: count, dtype: int64
count    800.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: IsUltraBeast, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

Có vẻ là không có Pokémon siêu thú nào trong bộ dữ liệu này, nên cột này có thể bị loại bỏ.

In [122]:
data = data.drop('IsUltraBeast', axis=1)

<span style = "font-family: Verdana; font-size: 20px">

##### **HasMega**
</span>

In [123]:
describe_feature(data, 'HasMega')

- Type: int64
- First rows:
0    0
1    0
2    1
3    1
4    0
Name: HasMega, dtype: int64
- Last rows:
795    1
796    1
797    0
798    0
799    0
Name: HasMega, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 2
- Unique value counts:
HasMega
0    696
1    104
Name: count, dtype: int64
count    800.000000
mean       0.130000
std        0.336514
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: HasMega, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

Thêm một cột để biểu thị Pokémon đó có ở trong dạng tiến hoá Mega hay không.

In [124]:
data['IsMega'] = 0
data.loc[data['Name'].str.startswith('Mega ', na=False), 'IsMega'] = 1

In [125]:
describe_feature(data, 'IsMega')

- Type: int64
- First rows:
0    0
1    0
2    0
3    1
4    0
Name: IsMega, dtype: int64
- Last rows:
795    0
796    1
797    0
798    0
799    0
Name: IsMega, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 2
- Unique value counts:
IsMega
0    752
1     48
Name: count, dtype: int64
count    800.000000
mean       0.060000
std        0.237635
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: IsMega, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **EvoStage**
</span>

In [126]:
describe_feature(data, 'EvoStage')

- Type: int64
- First rows:
0    1
1    2
2    3
3    3
4    1
Name: EvoStage, dtype: int64
- Last rows:
795    1
796    1
797    1
798    1
799    1
Name: EvoStage, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 3
- Unique value counts:
EvoStage
1    403
2    287
3    110
Name: count, dtype: int64
count    800.000000
mean       1.633750
std        0.712563
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        3.000000
Name: EvoStage, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

Cột này cho biết bậc tiến hoá hiện tại của mỗi Pokémon, giá trị có thể có là `1` (cho bậc tiến hoá đầu tiên), lần lượt là `2` và `3`. Pokémon không có tiến hoá sẽ có giá trị là `1`. Vấn đề có thể có là EvoStage có giá trị lớn hơn TotalEvoStages.

In [127]:
data.loc[data['EvoStage'] > data['TotalEvoStages']]

,ID,Name,Type 1,Type 2,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,LevelingRate,BaseFriendship,IsLegendary,IsMythical,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy,Ability 1,Ability 2,NoGender,EggGroup1,EggGroup2,IsMega
316,317,Shedinja,Bug,Ghost,None,3,1.0,90.0,45.0,30.0,30.0,40.0,236.0,1.2,0.8,0.5,Shed Pokémon,45,15,Erratic,70,0,0,0,3,2,Ninjask,0.0,0.0,2.0,0.5,0.5,2.0,0.5,2.0,1.0,2.0,1.0,0.5,1.0,1.0,1.0,1.0,2.0,1.0,Wonder Guard,None,1,Mineral,Bug,0


In [128]:
data.loc[data['EvoStage'] > data['TotalEvoStages'], 'EvoStage'] = data['TotalEvoStages']
data.loc[data['EvoStage'] > data['TotalEvoStages']]

,ID,Name,Type 1,Type 2,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,LevelingRate,BaseFriendship,IsLegendary,IsMythical,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy,Ability 1,Ability 2,NoGender,EggGroup1,EggGroup2,IsMega


<span style = "font-family: Verdana; font-size: 20px">

##### **TotalEvoStages**
</span>

In [129]:
describe_feature(data, 'TotalEvoStages')

- Type: int64
- First rows:
0    3
1    3
2    3
3    3
4    3
Name: TotalEvoStages, dtype: int64
- Last rows:
795    1
796    1
797    1
798    1
799    1
Name: TotalEvoStages, dtype: int64
- Number of missing values: 0
int64
- Number of distinct values: 3
- Unique value counts:
TotalEvoStages
2    370
3    286
1    144
Name: count, dtype: int64
count    800.000000
mean       2.177500
std        0.711777
min        1.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        3.000000
Name: TotalEvoStages, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **PreevoName**
</span>

In [130]:
describe_feature(data, 'PreevoName')

- Type: object
- First rows:
0    No Preevolution
1          Bulbasaur
2            Ivysaur
3           Venusaur
4    No Preevolution
Name: PreevoName, dtype: object
- Last rows:
795    No Preevolution
796            Diancie
797    No Preevolution
798    No Preevolution
799    No Preevolution
Name: PreevoName, dtype: object
- Number of missing values: 0
object
- Number of distinct values: 393
- Unique value counts:
PreevoName
No Preevolution           387
Eevee                       8
Tyrogue                     3
Mewtwo                      2
Wurmple                     2
Kirlia                      2
Poliwhirl                   2
Gloom                       2
Darumaka                    2
Espurr                      2
Doublade                    2
Slowpoke                    2
Snorunt                     2
Clamperl                    2
Charizard                   2
Dewott                      1
Pansear                     1
Magneton                    1
Sneasel                     1


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromNormal**
</span>

In [131]:
describe_feature(data, 'DamageFromNormal')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: DamageFromNormal, dtype: float64
- Last rows:
795    0.5
796    0.5
797    0.0
798    1.0
799    1.0
Name: DamageFromNormal, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 4
- Unique value counts:
DamageFromNormal
1.00    657
0.50     91
0.00     46
0.25      6
Name: count, dtype: int64
count    800.000000
mean       0.880000
std        0.275411
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: DamageFromNormal, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromFighting**
</span>

In [132]:
describe_feature(data, 'DamageFromFighting')

- Type: float64
- First rows:
0    0.5
1    0.5
2    0.5
3    0.5
4    1.0
Name: DamageFromFighting, dtype: float64
- Last rows:
795    1.0
796    1.0
797    0.0
798    1.0
799    1.0
Name: DamageFromFighting, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 7
- Unique value counts:
DamageFromFighting
1.00    321
0.50    192
2.00    182
0.00     46
0.25     44
4.00     14
1.50      1
Name: count, dtype: int64
count    800.000000
mean       1.061875
std        0.728412
min        0.000000
25%        0.500000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromFighting, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromFlying**
</span>

In [133]:
describe_feature(data, 'DamageFromFlying')

- Type: float64
- First rows:
0    2.0
1    2.0
2    2.0
3    2.0
4    1.0
Name: DamageFromFlying, dtype: float64
- Last rows:
795    0.5
796    0.5
797    1.0
798    1.0
799    1.0
Name: DamageFromFlying, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromFlying
1.00    489
2.00    175
0.50    116
4.00     11
0.25      9
Name: count, dtype: int64
count    800.000000
mean       1.179062
std        0.594522
min        0.250000
25%        1.000000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromFlying, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromPoison**
</span>

In [134]:
describe_feature(data, 'DamageFromPoison')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: DamageFromPoison, dtype: float64
- Last rows:
795    1.0
796    1.0
797    0.5
798    1.0
799    1.0
Name: DamageFromPoison, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 6
- Unique value counts:
DamageFromPoison
1.00    482
0.50    155
2.00     95
0.00     49
0.25     17
4.00      2
Name: count, dtype: int64
count    800.000000
mean       0.952187
std        0.510814
min        0.000000
25%        0.500000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromPoison, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromGround**
</span>

In [135]:
describe_feature(data, 'DamageFromGround')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    2.0
Name: DamageFromGround, dtype: float64
- Last rows:
795    2.0
796    2.0
797    1.0
798    1.0
799    2.0
Name: DamageFromGround, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 7
- Unique value counts:
DamageFromGround
1.00    400
2.00    190
0.00    104
0.50     87
4.00     12
0.25      6
1.50      1
Name: count, dtype: int64
count    800.000000
mean       1.093125
std        0.725399
min        0.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        4.000000
Name: DamageFromGround, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromRock**
</span>

In [136]:
describe_feature(data, 'DamageFromRock')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    2.0
Name: DamageFromRock, dtype: float64
- Last rows:
795    1.0
796    1.0
797    1.0
798    1.0
799    2.0
Name: DamageFromRock, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromRock
1.00    449
2.00    196
0.50    126
4.00     24
0.25      5
Name: count, dtype: int64
count    800.000000
mean       1.251562
std        0.703723
min        0.250000
25%        1.000000
50%        1.000000
75%        2.000000
max        4.000000
Name: DamageFromRock, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromBug**
</span>

In [137]:
describe_feature(data, 'DamageFromBug')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    0.5
Name: DamageFromBug, dtype: float64
- Last rows:
795    0.5
796    0.5
797    1.0
798    4.0
799    0.5
Name: DamageFromBug, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromBug
1.00    370
0.50    248
2.00    132
0.25     41
4.00      9
Name: count, dtype: int64
count    800.000000
mean       1.005313
std        0.610750
min        0.250000
25%        0.500000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromBug, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromGhost**
</span>

In [138]:
describe_feature(data, 'DamageFromGhost')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: DamageFromGhost, dtype: float64
- Last rows:
795    1.0
796    1.0
797    4.0
798    1.0
799    1.0
Name: DamageFromGhost, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromGhost
1.0    527
2.0    126
0.0    102
0.5     44
4.0      1
Name: count, dtype: int64
count    800.00000
mean       1.00625
std        0.55709
min        0.00000
25%        1.00000
50%        1.00000
75%        1.00000
max        4.00000
Name: DamageFromGhost, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromSteel**
</span>

In [139]:
describe_feature(data, 'DamageFromSteel')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    0.5
Name: DamageFromSteel, dtype: float64
- Last rows:
795    4.00
796    4.00
797    1.00
798    1.00
799    0.25
Name: DamageFromSteel, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromSteel
1.00    450
0.50    239
2.00     96
0.25     10
4.00      5
Name: count, dtype: int64
count    800.00000
mean       0.98000
std        0.50783
min        0.25000
25%        0.50000
50%        1.00000
75%        1.00000
max        4.00000
Name: DamageFromSteel, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromFire**
</span>

In [140]:
describe_feature(data, 'DamageFromFire')

- Type: float64
- First rows:
0    2.0
1    2.0
2    2.0
3    1.0
4    0.5
Name: DamageFromFire, dtype: float64
- Last rows:
795    0.50
796    0.50
797    1.00
798    1.00
799    0.25
Name: DamageFromFire, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 6
- Unique value counts:
DamageFromFire
1.00    353
0.50    228
2.00    182
0.25     18
4.00     18
1.50      1
Name: count, dtype: int64
count    800.000000
mean       1.136250
std        0.704468
min        0.250000
25%        0.500000
50%        1.000000
75%        1.625000
max        4.000000
Name: DamageFromFire, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromWater**
</span>

In [141]:
describe_feature(data, 'DamageFromWater')

- Type: float64
- First rows:
0    0.5
1    0.5
2    0.5
3    0.5
4    2.0
Name: DamageFromWater, dtype: float64
- Last rows:
795    2.0
796    2.0
797    1.0
798    1.0
799    1.0
Name: DamageFromWater, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromWater
1.00    430
0.50    224
2.00    126
4.00     14
0.25      6
Name: count, dtype: int64
count    800.000000
mean       1.064375
std        0.620932
min        0.250000
25%        0.500000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromWater, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromGrass**
</span>

In [142]:
describe_feature(data, 'DamageFromGrass')

- Type: float64
- First rows:
0    0.25
1    0.25
2    0.25
3    0.25
4    0.50
Name: DamageFromGrass, dtype: float64
- Last rows:
795    2.0
796    2.0
797    1.0
798    1.0
799    1.0
Name: DamageFromGrass, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromGrass
1.00    298
0.50    255
2.00    130
0.25     88
4.00     29
Name: count, dtype: int64
count    800.000000
mean       1.029375
std        0.793875
min        0.250000
25%        0.500000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromGrass, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromElectric**
</span>

In [143]:
describe_feature(data, 'DamageFromElectric')

- Type: float64
- First rows:
0    0.5
1    0.5
2    0.5
3    0.5
4    1.0
Name: DamageFromElectric, dtype: float64
- Last rows:
795    1.0
796    1.0
797    1.0
798    1.0
799    2.0
Name: DamageFromElectric, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 6
- Unique value counts:
DamageFromElectric
1.00    397
2.00    172
0.50    152
0.00     68
4.00      8
0.25      3
Name: count, dtype: int64
count    800.000000
mean       1.062188
std        0.660521
min        0.000000
25%        0.500000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromElectric, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromPsychic**
</span>

In [144]:
describe_feature(data, 'DamageFromPsychic')

- Type: float64
- First rows:
0    2.0
1    2.0
2    2.0
3    2.0
4    1.0
Name: DamageFromPsychic, dtype: float64
- Last rows:
795    1.0
796    1.0
797    0.5
798    0.0
799    1.0
Name: DamageFromPsychic, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 6
- Unique value counts:
DamageFromPsychic
1.00    534
0.50    111
2.00     96
0.00     50
0.25      7
4.00      2
Name: count, dtype: int64
count    800.000000
mean       0.989062
std        0.494768
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromPsychic, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromIce**
</span>

In [145]:
describe_feature(data, 'DamageFromIce')

- Type: float64
- First rows:
0    2.0
1    2.0
2    2.0
3    1.0
4    0.5
Name: DamageFromIce, dtype: float64
- Last rows:
795    1.00
796    1.00
797    1.00
798    1.00
799    0.25
Name: DamageFromIce, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromIce
1.00    351
2.00    208
0.50    206
4.00     26
0.25      9
Name: count, dtype: int64
count    800.000000
mean       1.220312
std        0.758541
min        0.250000
25%        0.500000
50%        1.000000
75%        2.000000
max        4.000000
Name: DamageFromIce, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromDragon**
</span>

In [146]:
describe_feature(data, 'DamageFromDragon')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: DamageFromDragon, dtype: float64
- Last rows:
795    0.0
796    0.0
797    1.0
798    1.0
799    1.0
Name: DamageFromDragon, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 4
- Unique value counts:
DamageFromDragon
1.0    667
2.0     48
0.5     45
0.0     40
Name: count, dtype: int64
count    800.000000
mean       0.981875
std        0.351978
min        0.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        2.000000
Name: DamageFromDragon, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromDark**
</span>

In [147]:
describe_feature(data, 'DamageFromDark')

- Type: float64
- First rows:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: DamageFromDark, dtype: float64
- Last rows:
795    0.5
796    0.5
797    4.0
798    1.0
799    1.0
Name: DamageFromDark, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromDark
1.00    561
0.50    119
2.00    116
0.25      3
4.00      1
Name: count, dtype: int64
count    800.000000
mean       1.071562
std        0.436651
min        0.250000
25%        1.000000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromDark, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

##### **DamageFromFairy**
</span>

In [148]:
describe_feature(data, 'DamageFromFairy')

- Type: float64
- First rows:
0    0.5
1    0.5
2    0.5
3    0.5
4    0.5
Name: DamageFromFairy, dtype: float64
- Last rows:
795    1.0
796    1.0
797    1.0
798    2.0
799    0.5
Name: DamageFromFairy, dtype: float64
- Number of missing values: 0
float64
- Number of distinct values: 5
- Unique value counts:
DamageFromFairy
1.00    527
0.50    149
2.00    117
4.00      6
0.25      1
Name: count, dtype: int64
count    800.000000
mean       1.074688
std        0.505724
min        0.250000
25%        1.000000
50%        1.000000
75%        1.000000
max        4.000000
Name: DamageFromFairy, dtype: float64


<span style = "font-family: Verdana; font-size: 20px">

#### **II. Combat dataset**
</span>

In [149]:
print("Combat dataset")
print(f"Number of rows: {combats.shape[0]}, number of columns: {combats.shape[1]}")
print(f"Column names: {combats.columns}")
print(f"Number of missing values: {combats.isna().sum().sum()}")

Combat dataset
Number of rows: 50000, number of columns: 3
Column names: Index(['First_pokemon', 'Second_pokemon', 'Winner'], dtype='object')
Number of missing values: 0


In [150]:
# Tổng số trận thắng của mỗi Pokémon
total_wins = combats['Winner'].value_counts()
# Số trận thắng của mỗi Pokémon
number_of_wins = combats.groupby('Winner').count()

countByFirst = combats.groupby('Second_pokemon').count() # Hai kiểu group cho ra cùng kết quả
countBySecond = combats.groupby('First_pokemon').count()
print("Looking at the dimensions of our dataframes")
print("Count by first winner shape: " + str(countByFirst.shape))
print("Count by second winner shape: " + str(countBySecond.shape))
print("Total wins shape : " + str(total_wins.shape))

Looking at the dimensions of our dataframes
Count by first winner shape: (784, 2)
Count by second winner shape: (784, 2)
Total wins shape : (783,)


<span style = "font-family: Verdana; font-size: 20px">

Có thể thấy số chiều của dataframe tổng trận thắng không giống với số chiều của hai dataframe đếm số trận thắng theo từng Pokémon. 

Điều này cho thấy có một Pokémon chưa từng thắng trận nào trong dữ liệu.

In [151]:
locate_losing_pokemon = np.setdiff1d(countByFirst.index.values, number_of_wins.index.values)-1
losing_pokemon = data.iloc[locate_losing_pokemon[0],]
losing_pokemon

ID                                   231
Name                             Shuckle
Type 1                               Bug
Type 2                              Rock
HiddenAbility                   Contrary
Generation                             2
Hp                                  20.0
Attack                              10.0
Defense                            230.0
SpecialAttack                       10.0
SpecialDefense                     230.0
Speed                                5.0
TotalStats                         505.0
Weight                              20.5
Height                               0.6
GenderProbM                          0.5
Category                    Mold Pokémon
CatchRate                            190
EggCycles                             20
LevelingRate                 Medium Slow
BaseFriendship                        70
IsLegendary                            0
IsMythical                             0
HasMega                                0
EvoStage        

<span style = "font-family: Verdana; font-size: 20px">

Tội nghiệp Shuckle 🙁 Tuy chỉ số phòng thủ khá ấn tượng, cao nhất trong số các Pokémon, nhưng các chỉ số khác lại cực kỳ thấp so với mặt bằng chung.

<span style = "font-family: Verdana; font-size: 20px">

Có tồn tại Pokémon chưa từng thắng trận nào, vậy chắc cũng phải có Pokémon nào chưa từng tham gia trận nào chứ?

In [152]:
pokemon_not_in_combat = data[~data['ID'].astype(int).isin(number_of_wins.index)]
pokemon_not_in_combat

,ID,Name,Type 1,Type 2,HiddenAbility,Generation,Hp,Attack,Defense,SpecialAttack,SpecialDefense,Speed,TotalStats,Weight,Height,GenderProbM,Category,CatchRate,EggCycles,LevelingRate,BaseFriendship,IsLegendary,IsMythical,HasMega,EvoStage,TotalEvoStages,PreevoName,DamageFromNormal,DamageFromFighting,DamageFromFlying,DamageFromPoison,DamageFromGround,DamageFromRock,DamageFromBug,DamageFromGhost,DamageFromSteel,DamageFromFire,DamageFromWater,DamageFromGrass,DamageFromElectric,DamageFromPsychic,DamageFromIce,DamageFromDragon,DamageFromDark,DamageFromFairy,Ability 1,Ability 2,NoGender,EggGroup1,EggGroup2,IsMega
11,12,Blastoise,Water,Water,Rain Dish,1,79.0,83.0,100.0,85.0,105.0,78.0,530.0,85.5,1.6,0.875,Shellfish Pokémon,45,20,Medium Slow,70,0,0,1,3,3,Wartortle,1.0,1.00,1.00,1.0,1.0,1.0,1.0,1.0,0.50,0.5,0.5,2.00,2.0,1.0,0.5,1.0,1.0,1.0,Torrent,None,0,Monster,Water 1,0
32,33,Sandshrew,Ground,Ground,Sand Rush,1,50.0,75.0,85.0,20.0,30.0,40.0,300.0,12.0,0.6,0.500,Mouse Pokémon,255,20,Medium Fast,70,0,0,0,1,2,No Preevolution,1.0,1.00,1.00,0.5,1.0,0.5,1.0,1.0,1.00,1.0,2.0,2.00,0.0,1.0,2.0,1.0,1.0,1.0,Sand Veil,None,0,Field,No Eggs Discovered,0
45,46,Wigglytuff,Normal,Fairy,Frisk,1,140.0,70.0,45.0,85.0,50.0,45.0,435.0,12.0,1.0,0.250,Balloon Pokémon,50,10,Fast,70,0,0,0,3,3,Jigglypuff,1.0,1.00,1.00,2.0,1.0,1.0,0.5,0.0,2.00,1.0,1.0,1.00,1.0,1.0,1.0,0.0,0.5,1.0,Cute Charm,Competitive,0,Fairy,No Eggs Discovered,0
65,66,Poliwag,Water,Water,Swift Swim,1,40.0,50.0,40.0,40.0,40.0,90.0,300.0,12.4,0.6,0.500,Tadpole Pokémon,255,20,Medium Slow,70,0,0,0,1,3,No Preevolution,1.0,1.00,1.00,1.0,1.0,1.0,1.0,1.0,0.50,0.5,0.5,2.00,2.0,1.0,0.5,1.0,1.0,1.0,Water Absorb,Damp,0,Water 1,No Eggs Discovered,0
77,78,Victreebel,Grass,Poison,Gluttony,1,80.0,105.0,65.0,100.0,70.0,70.0,490.0,15.5,1.7,0.500,Flycatcher Pokémon,45,20,Medium Slow,70,0,0,0,3,3,Weepinbell,1.0,0.50,2.00,1.0,1.0,1.0,1.0,1.0,1.00,2.0,0.5,0.25,0.5,2.0,2.0,1.0,1.0,0.5,Chlorophyll,None,0,Grass,No Eggs Discovered,0
89,90,Magneton,Electric,Steel,Analytic,1,50.0,60.0,95.0,120.0,70.0,70.0,465.0,60.0,1.0,0.500,Magnet Pokémon,60,20,Medium Fast,70,0,0,0,2,3,Magnemite,0.5,2.00,0.25,0.0,4.0,0.5,0.5,1.0,0.25,2.0,1.0,0.50,0.5,0.5,0.5,0.5,1.0,0.5,Magnet Pull,Sturdy,1,Mineral,No Eggs Discovered,0
143,144,Ditto,Normal,Normal,Imposter,1,48.0,48.0,48.0,48.0,48.0,48.0,288.0,4.0,0.3,0.500,Transform Pokémon,35,20,Medium Fast,70,0,0,0,1,1,No Preevolution,1.0,2.00,1.00,1.0,1.0,1.0,1.0,0.0,1.00,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,Limber,None,1,Ditto,No Eggs Discovered,0
182,183,Ariados,Bug,Poison,Sniper,2,70.0,90.0,70.0,60.0,70.0,40.0,400.0,33.5,1.1,0.500,Long Leg Pokémon,90,15,Fast,70,0,0,0,2,2,Spinarak,1.0,0.25,2.00,0.5,1.0,2.0,0.5,1.0,1.00,2.0,1.0,0.25,1.0,2.0,1.0,1.0,1.0,0.5,Swarm,Insomnia,0,Bug,No Eggs Discovered,0
230,231,Shuckle,Bug,Rock,Contrary,2,20.0,10.0,230.0,10.0,230.0,5.0,505.0,20.5,0.6,0.500,Mold Pokémon,190,20,Medium Slow,70,0,0,0,1,1,No Preevolution,0.5,1.00,1.00,0.5,1.0,2.0,1.0,1.0,2.00,1.0,2.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,Sturdy,Gluttony,0,Bug,No Eggs Discovered,0
235,236,Ursaring,Normal,Normal,Unnerve,2,90.0,130.0,75.0,75.0,75.0,55.0,500.0,125.8,1.8,0.500,Hibernator Pokémon,60,20,Medium Fast,70,0,0,0,2,3,Teddiursa,1.0,2.00,1.00,1.0,1.0,1.0,1.0,0.0,1.00,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,Guts,Quick Feet,0,Field,No Eggs Discovered,0


<span style = "font-family: Verdana; font-size: 20px">

Vậy linh cảm trước đó là đúng, thực sự có Pokémon chưa từng tham gia trận nào.

<span style = "font-family: Verdana; font-size: 20px">

Tính toán tỷ lệ thắng `WinRate`, tổng trận tham gia `TotalFights` và trận thắng `FightsWon` cho mỗi Pokémon, sẽ rất hữu ích cho giai đoạn trực quan hoá sắp tới, có thể coi là một phần của feature engineering luôn.
</span>

In [153]:
number_of_wins = number_of_wins.sort_index()
number_of_wins.rename(columns={'First_pokemon': 'FightsWon'}, inplace=True)
number_of_wins = number_of_wins.drop('Second_pokemon', axis = 1)
number_of_wins['TotalFights'] = countByFirst.Winner + countBySecond.Winner

In [154]:
number_of_wins.info()

<class 'pandas.core.frame.DataFrame'>
Index: 783 entries, 1 to 800
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   FightsWon    783 non-null    int64
 1   TotalFights  783 non-null    int64
dtypes: int64(2)
memory usage: 18.4 KB


In [155]:
data['ID'] = data['ID'].astype(int) # Chuyển ID sang int để merge với number_of_wins

In [156]:
data = pd.merge(data, number_of_wins, left_on='ID', right_index = True, how='left')

<span style = "font-family: Verdana; font-size: 20px">

Xử lý định dạng và missing value cho các cột mới được thêm. 

In [157]:
data['FightsWon'] = data['FightsWon'].fillna(0).astype(int)
data['TotalFights'] = data['TotalFights'].fillna(0).astype(int)
data['WinPercentage'] = (data['FightsWon']/data['TotalFights']).round(6)
data['WinPercentage'] = data['WinPercentage'].fillna(0.0)

In [158]:
with open('data/modified/viz_pokemon.csv', 'w', encoding = 'utf-8') as f:
    data.to_csv(f, index=False)